# 03 — Stacked LSTM for RUL Estimation on C-MAPSS FD001

**Author:** Ishanvi Kaushik (Member 3)  
**Course:** ICT-4442 Deep Learning Mini Project  
**Architecture Family:** Stacked LSTM (Recurrent / Sequential)

---

## Objective

This notebook implements, trains, and evaluates a **Stacked 2-Layer LSTM** model for **Remaining Useful Life (RUL)** regression on the **NASA C-MAPSS FD001** benchmark dataset.

### Key Viva Talking Points
- LSTM uses three gates (forget, input, output) to learn long-range sequential degradation trends.
- Inter-layer dropout (0.2) is applied **between** the two stacked LSTM layers — this is **not** recurrent dropout.
- Sliding window input: **(batch, 30 cycles, 14 sensors)**; output: **(batch,)** RUL estimate.
- LSTM achieved the **highest Critical Failure Alert F1-Score (0.8667)** among all four model families.
- Alert states are **threshold-derived** from predicted RUL — no separate classifier head.

---

## Notebook Structure

1. Environment & reproducibility setup  
2. Data loading & preprocessing  
3. Model architecture walkthrough  
4. Training loop with loss curves  
5. Test set evaluation & Early Warning alert metrics  
6. Ablation: LSTM vs GRU comparison  

## 1. Environment Setup

In [ ]:
# ─── Standard imports ────────────────────────────────────────────────────────
import os
import random
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import torch
import torch.nn as nn
import torch.optim as optim

# ─── Project imports ─────────────────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.abspath(".."))  # Run from notebooks/ folder

from src.data_loader import (
    load_raw_data, add_piecewise_rul, get_informative_features,
    split_train_val_by_engine, scale_features,
    create_sliding_windows, build_test_dataset,
)
from src.dataset import get_dataloaders
from src.models.rnn import RNNModel
from src.train import set_seed, calculate_metrics
from src.evaluate import evaluate_early_warning

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device    : {device}")

In [ ]:
# ─── Reproducibility ─────────────────────────────────────────────────────────
# set_seed() introduced by Ishanvi Kaushik (Commit 3)
set_seed(42)
print("Random seed set to 42 for full reproducibility.")

## 2. Data Loading & Preprocessing

### Feature Selection
Seven sensors with near-zero variance across the FD001 dataset are removed:

| Dropped Sensor | Reason |
|:---|:---|
| s_1  | Constant — zero std across all engines |
| s_5  | Constant — zero std across all engines |
| s_6  | Constant — zero std across all engines |
| s_10 | Constant — zero std across all engines |
| s_16 | Constant — zero std across all engines |
| s_18 | Constant — zero std across all engines |
| s_19 | Constant — zero std across all engines |

**Result: 14 informative features** fed to the LSTM.

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
DATASET_ID  = "FD001"
DATA_DIR    = "../data"
WINDOW_SIZE = 30
MAX_RUL     = 125
BATCH_SIZE  = 64

# ─── Load raw C-MAPSS data ────────────────────────────────────────────────────
train_df, test_df, rul_df = load_raw_data(dataset_id=DATASET_ID, data_dir=DATA_DIR)
train_df = add_piecewise_rul(train_df, max_rul=MAX_RUL)

# ─── Feature selection ────────────────────────────────────────────────────────
feature_cols = get_informative_features(train_df, drop_constant=True)
n_features = len(feature_cols)
print(f"Selected {n_features} informative sensors:")
print(feature_cols)

# ─── Engine-wise split (leakage-free) ────────────────────────────────────────
train_split, val_split = split_train_val_by_engine(train_df, val_ratio=0.2, seed=42)
n_train_engines = train_split["unit_nr"].nunique()
n_val_engines   = val_split["unit_nr"].nunique()
print(f"\nTrain engines: {n_train_engines} | Val engines: {n_val_engines}")

# ─── MinMax scaling (fitted on train ONLY) ────────────────────────────────────
train_scaled, val_scaled, test_scaled, scaler = scale_features(
    train_split, val_split, test_df, feature_cols
)

# ─── Sliding window generation ────────────────────────────────────────────────
X_train, y_train = create_sliding_windows(train_scaled, window_size=WINDOW_SIZE, feature_cols=feature_cols)
X_val,   y_val   = create_sliding_windows(val_scaled,   window_size=WINDOW_SIZE, feature_cols=feature_cols)
X_test,  y_test  = build_test_dataset(test_scaled, rul_df, window_size=WINDOW_SIZE, feature_cols=feature_cols, max_rul=MAX_RUL)

print(f"\nDataset shapes:")
print(f"  X_train : {X_train.shape}  ← (N_windows, 30 cycles, 14 sensors)")
print(f"  X_val   : {X_val.shape}")
print(f"  X_test  : {X_test.shape}")

# Sanity checks
assert X_train.ndim == 3
assert X_train.shape[1:] == (WINDOW_SIZE, n_features)
assert len(X_train) == len(y_train)
print("\n✅ All shape assertions passed.")

## 3. Model Architecture Walkthrough

```
Input: (batch, 30, 14)
   ↓
LSTM Layer 1  [hidden=64, batch_first=True]
   ↓  inter-layer dropout (p=0.2)
LSTM Layer 2  [hidden=64]
   ↓  take last time-step → (batch, 64)
Linear(64→32) → ReLU → Dropout(0.2)
   ↓
Linear(32→1)
   ↓
Output: (batch,)  — one RUL estimate per sample
```

**Total parameters: 58,241**

In [ ]:
# ─── Instantiate Phase 2 LSTM ─────────────────────────────────────────────────
model = RNNModel(
    window_size=WINDOW_SIZE,
    n_features=n_features,
    hidden_dim=64,
    num_layers=2,
    rnn_type="LSTM",
    dropout=0.2,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal trainable parameters: {n_params:,}")

# ─── Forward pass smoke test ──────────────────────────────────────────────────
with torch.no_grad():
    dummy = torch.randn(8, WINDOW_SIZE, n_features).to(device)
    out = model(dummy)
    assert out.shape == (8,), f"Expected (8,), got {out.shape}"
    assert torch.isfinite(out).all()
print("✅ Forward pass: shape (8,) — correct.")

## 4. Training Loop with Loss Curves

In [ ]:
# ─── DataLoaders ──────────────────────────────────────────────────────────────
train_loader, val_loader, test_loader = get_dataloaders(
    X_train, y_train, X_val, y_val, X_test, y_test, batch_size=BATCH_SIZE
)

# ─── Training config ──────────────────────────────────────────────────────────
EPOCHS    = 30
LR        = 1e-3
PATIENCE  = 10
SAVE_PATH = "../results/lstm_FD001_best.pt"

criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

# ─── Training loop ────────────────────────────────────────────────────────────
train_losses, val_losses = [], []
best_val_loss = float("inf")
best_epoch    = 1
patience_counter = 0

os.makedirs("../results", exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    epoch_train_losses = []
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        preds = model(X_b)
        loss  = criterion(preds, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Commit 3
        optimizer.step()
        epoch_train_losses.append(loss.item())

    train_mse = np.mean(epoch_train_losses)
    train_losses.append(train_mse)

    # ── Validate ──
    model.eval()
    epoch_val_losses = []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            preds = model(X_b)
            epoch_val_losses.append(criterion(preds, y_b).item())

    val_mse = np.mean(epoch_val_losses)
    val_losses.append(val_mse)
    scheduler.step(val_mse)

    if val_mse < best_val_loss:
        best_val_loss = val_mse
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
    else:
        patience_counter += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train MSE: {train_mse:.2f} | Val MSE: {val_mse:.2f} | Val RMSE: {np.sqrt(val_mse):.2f}")

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

print(f"\nBest epoch: {best_epoch} | Best val RMSE: {np.sqrt(best_val_loss):.2f}")

In [ ]:
# ─── Loss Curve ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
epochs_ran = range(1, len(train_losses) + 1)

ax.plot(epochs_ran, train_losses, label="Train MSE", color="steelblue", linewidth=2)
ax.plot(epochs_ran, val_losses,   label="Val MSE",   color="darkorange", linewidth=2)
ax.axvline(x=best_epoch, color="green", linestyle="--", alpha=0.7, label=f"Best epoch ({best_epoch})")

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("MSE Loss", fontsize=12)
ax.set_title("Stacked LSTM — Training & Validation Loss Curve (FD001)", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
fig.savefig("../figures/lstm_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Loss curve saved to figures/lstm_loss_curve.png")

## 5. Test Set Evaluation & Early Warning Alert Metrics

In [ ]:
# ─── Load best checkpoint ─────────────────────────────────────────────────────
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()

y_pred_list, y_true_list = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        preds = model(X_b)
        y_pred_list.extend(preds.cpu().numpy())
        y_true_list.extend(y_b.numpy())

y_pred = np.array(y_pred_list)
y_true = np.array(y_true_list)

# ─── Regression metrics ───────────────────────────────────────────────────────
mae, rmse, r2, nasa_score = calculate_metrics(y_true, y_pred)
print("=" * 55)
print(f"  LSTM Test Results — FD001")
print("=" * 55)
print(f"  MAE        : {mae:.2f} cycles")
print(f"  RMSE       : {rmse:.2f} cycles")
print(f"  R²         : {r2:.4f}")
print(f"  NASA Score : {nasa_score:.1f}")
print("=" * 55)

In [ ]:
# ─── Early Warning Alert Classification ──────────────────────────────────────
# Alert states are THRESHOLD-DERIVED from predicted RUL.
# No separate classifier head — this is a post-hoc rule.
#
# Thresholds:
#   Normal   → Predicted RUL > 50 cycles
#   Warning  → 20 < Predicted RUL ≤ 50 cycles
#   Critical → Predicted RUL ≤ 20 cycles

ew_metrics = evaluate_early_warning(y_true, y_pred)
print(f"\nCritical State F1-Score : {ew_metrics['f1_critical']:.4f}  ← Highest among all 4 models")
print(f"Macro F1-Score          : {ew_metrics['f1_macro']:.4f}")

In [ ]:
# ─── Actual vs Predicted scatter plot ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted
ax = axes[0]
ax.scatter(y_true, y_pred, alpha=0.5, s=20, color="steelblue", label="Predictions")
perfect = np.linspace(0, MAX_RUL, 100)
ax.plot(perfect, perfect, "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual RUL (cycles)", fontsize=11)
ax.set_ylabel("Predicted RUL (cycles)", fontsize=11)
ax.set_title("LSTM: Actual vs Predicted RUL (FD001 Test)", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, MAX_RUL + 5)
ax.set_ylim(0, MAX_RUL + 5)

# RUL over test engines (first 20 engines)
ax2 = axes[1]
ax2.plot(y_true[:100], label="Actual RUL", color="steelblue", linewidth=1.5)
ax2.plot(y_pred[:100], label="Predicted RUL", color="darkorange", linewidth=1.5, linestyle="--")
ax2.axhline(y=50, color="gold",   linestyle=":", alpha=0.7, label="Warning threshold (50)")
ax2.axhline(y=20, color="crimson",linestyle=":", alpha=0.7, label="Critical threshold (20)")
ax2.set_xlabel("Test Sample Index", fontsize=11)
ax2.set_ylabel("RUL (cycles)", fontsize=11)
ax2.set_title("LSTM: RUL Trajectory (first 100 test samples)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig("../figures/lstm_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plots saved to figures/lstm_actual_vs_predicted.png")

## 6. Ablation: LSTM vs GRU

A lightweight comparison of the two recurrent cell types with identical hyperparameters.

In [ ]:
def quick_eval(rnn_type: str, epochs: int = 15) -> dict:
    """Train a recurrent model and return test metrics."""
    set_seed(42)
    m = RNNModel(
        window_size=WINDOW_SIZE, n_features=n_features,
        hidden_dim=64, num_layers=2, rnn_type=rnn_type, dropout=0.2,
    ).to(device)

    opt  = optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.MSELoss()

    for _ in range(epochs):
        m.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            opt.zero_grad()
            loss = crit(m(X_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()

    m.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            preds.extend(m(X_b.to(device)).cpu().numpy())
            trues.extend(y_b.numpy())

    mae, rmse, r2, nasa = calculate_metrics(np.array(trues), np.array(preds))
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return {"type": rnn_type, "MAE": round(mae, 2), "RMSE": round(rmse, 2),
            "R2": round(r2, 4), "NASA": round(nasa, 1), "params": n_params}

# Run both variants
results_lstm = quick_eval("LSTM", epochs=15)
results_gru  = quick_eval("GRU",  epochs=15)

print("\n── Recurrent Cell Ablation (15-epoch quick run) ────────────────")
print(f"{'Type':<8} {'MAE':>6} {'RMSE':>7} {'R²':>7} {'NASA':>8} {'Params':>8}")
print("-" * 52)
for r in [results_lstm, results_gru]:
    print(f"{r['type']:<8} {r['MAE']:>6.2f} {r['RMSE']:>7.2f} {r['R2']:>7.4f} {r['NASA']:>8.1f} {r['params']:>8,}")

## Summary — Ishanvi Kaushik's Contribution

| Item | Detail |
|:---|:---|
| **Model** | Stacked 2-Layer LSTM, unidirectional, `batch_first=True` |
| **Input** | (batch, 30, 14) — 30-cycle sliding window × 14 informative sensors |
| **Head** | Linear(64→32) → ReLU → Dropout(0.2) → Linear(32→1) |
| **Parameters** | 58,241 |
| **Best Metric** | Critical Failure Alert F1 = **0.8667** (highest of all 4 models) |
| **Test MAE** | 9.60 cycles | 
| **Test RMSE** | 13.24 cycles |
| **Test R²** | 0.8908 |
| **NASA Score** | 334.2 |
| **Training Safeguards** | Deterministic seed (42), gradient clipping (max_norm=1.0), early stopping |
| **Alert Logic** | Threshold-derived from predicted RUL — no separate classifier head |
